In [29]:
import geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from splot.esda import moran_scatterplot, lisa_cluster
import rasterio
import numpy as np
import glob
import os
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
from shapely.geometry import Point
from libpysal.weights import lag_spatial
from esda.moran import Moran_Local
import matplotlib.patches as mpatches
from shapely.geometry import box
from shapely import make_valid
import rasterio.mask
from pyproj import Transformer
from rasterio.warp import calculate_default_transform, reproject, Resampling
import os
import pandas as pd


In [40]:
def create_grid_over_shapefile(shapefile_path, cell_size, output_path=None):
    # Leer shapefile
    gdf = gpd.read_file(shapefile_path)
    
    # Reproyectar a una proyección métrica si está en grados (WGS84)
    #if gdf.crs.to_string() == 'EPSG:4326':
    gdf = gdf.to_crs(epsg=32612)  # Web Mercator para trabajar en metros

    # Calcular el bounding box total
    bounds = gdf.total_bounds  # (minx, miny, maxx, maxy)

    minx, miny, maxx, maxy = bounds

    # Crear celdas
    rows = int((maxy - miny) // cell_size) + 1
    cols = int((maxx - minx) // cell_size) + 1

    grid_cells = []
    for i in range(cols):
        for j in range(rows):
            x1 = minx + i * cell_size
            y1 = miny + j * cell_size
            x2 = x1 + cell_size
            y2 = y1 + cell_size
            cell = box(x1, y1, x2, y2)
            grid_cells.append(cell)

    grid = gpd.GeoDataFrame({'geometry': grid_cells}, crs=gdf.crs)

    # Recortar la cuadrícula al shapefile original (opcional)
    clipped_grid = gpd.overlay(grid, gdf, how='intersection')

    if output_path:
        clipped_grid.to_file(output_path)
        print(f"Cuadrícula guardada en: {output_path}")
    return clipped_grid

def promedio_por_celda(ruta_raster, grid, nombre_variable="valor"):
    valores = []
    with rasterio.open(ruta_raster) as src:
        for geom in grid.geometry:
            try:
                out_image, out_transform = rasterio.mask.mask(src, [geom], crop=True)
                data = out_image[0]
                data = data.astype('float32')
                nodata = src.nodata
                if nodata is not None:
                    data[data == nodata] = np.nan
                media = np.nanmean(data)
            except Exception as e:
                media = np.nan
            valores.append(media)

    grid[nombre_variable] = valores
    return grid

def reproject_image(input_raster, output_raster, nuevo_crs):
    with rasterio.open(input_raster) as src:
        transform, width, height = calculate_default_transform(
            src.crs, nuevo_crs, src.width, src.height, *src.bounds
        )
        
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': nuevo_crs,
            'transform': transform,
            'width': width,
            'height': height
        })

        with rasterio.open(output_raster, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):  # Para cada banda
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=nuevo_crs,
                    resampling=Resampling.nearest  # O bilinear, cubic, etc.
                )


### Crear grid de celdas 500x500 sobre Yellowstone

In [46]:
grid = create_grid_over_shapefile('datos/YELL_tracts/YELL_boundary.shp', 500, 'datos/grids/grid_yell_500.shp')

Cuadrícula guardada en: datos/grids/grid_yell_500.shp


/Users/ruthparajo/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:709: RuntimeWarning: Field DATE_EDIT create as date field, though DateTime requested.
  ogr_write(


### Hacer media ndvi 2001-2013 y 2014-2024

In [23]:
rutas_tifs = []
ruta_salida = 'datos/ndvi_media_2001_2013.tif'
for year in range(2001, 2014):
    rutas_tifs.append(os.path.join('datos/ndvi_anual/', f"{year}_NDVI_MNDWI.tif"))
with rasterio.open(rutas_tifs[0]) as src0:
    meta = src0.meta.copy()
    data_stack = []

# Leer todos los tifs y apilarlos
for ruta in rutas_tifs:
    with rasterio.open(ruta) as src:
        data = src.read(1).astype('float32')  # Leer la banda 1
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        data_stack.append(data)

# Convertir lista en array 3D: (num_tifs, height, width)
stack_array = np.stack(data_stack)

# Calcular la media ignorando NaNs
media = np.nanmean(stack_array, axis=0)

# Actualizar nodata y tipo de datos
meta.update(dtype='float32', count=1, nodata=np.nan)

# Escribir el resultado
with rasterio.open(ruta_salida, 'w', **meta) as dst:
    dst.write(media, 1)

print(f"TIFF de media guardado en: {ruta_salida}")

TIFF de media guardado en: datos/ndvi_media_2001_2013.tif


In [24]:
rutas_tifs = []
ruta_salida = 'datos/ndvi_media_2014_2024_final.tif'
for year in range(2014, 2025):
    rutas_tifs.append(os.path.join('datos/ndvi_anual/', f"{year}_NDVI_MNDWI.tif"))
with rasterio.open(rutas_tifs[0]) as src0:
    meta = src0.meta.copy()
    data_stack = []

# Leer todos los tifs y apilarlos
for ruta in rutas_tifs:
    with rasterio.open(ruta) as src:
        data = src.read(1).astype('float32')  # Leer la banda 1
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
        data_stack.append(data)

# Convertir lista en array 3D: (num_tifs, height, width)
stack_array = np.stack(data_stack)

# Calcular la media ignorando NaNs
media = np.nanmean(stack_array, axis=0)

# Actualizar nodata y tipo de datos
meta.update(dtype='float32', count=1, nodata=np.nan)

# Escribir el resultado
with rasterio.open(ruta_salida, 'w', **meta) as dst:
    dst.write(media, 1)

print(f"TIFF de media guardado en: {ruta_salida}")

TIFF de media guardado en: datos/ndvi_media_2014_2024_final.tif


In [19]:
reproject_image('datos/ndvi_media_2001_2013.tif', 'datos/ndvi_media_2001_2013_reprojected.tif', 'EPSG:32612')
reproject_image('datos/ndvi_media_2014_2024.tif', 'datos/ndvi_media_2014_2024_reprojected.tif', 'EPSG:32612')

## Reproyectar LST

In [53]:
reproject_image('lst/lst_2001_2013.tif', 'datos/lst_reprojected_2001_2013.tif', 'EPSG:32612')
reproject_image('lst/lst_2013_2024.tif', 'datos/lst_reprojected_2014_2024.tif', 'EPSG:32612')

## Reproyectar NLCD

In [54]:
reproject_image('ncld/NLCD_2011_RGB.tif', 'datos/NLCD_reprojected_2011.tif', 'EPSG:32612')
reproject_image('ncld/NLCD_2021_RGB.tif', 'datos/NLCD_reprojected_2021.tif', 'EPSG:32612')

## Presencia de lobos

In [25]:
p = "1995_2022-mcps/1995_2007_mcps/2001_all.shp"
pck_2001 = gpd.read_file(p)

In [31]:
grid['presencia_lobos'] = grid.geometry.intersects(pck_2001.unary_union).astype(int)


/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/933241890.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  grid['presencia_lobos'] = grid.geometry.intersects(pck_2001.unary_union).astype(int)


In [ ]:
archivos = sorted(glob.glob(os.path.join('datos/1995_2022-mcps/2022_Wolf Territory Shapefiles/mcp_outputs/', "*.shp")))
gdfs = [gpd.read_file(shp).assign(source=os.path.basename(shp)) for shp in archivos]
mcp_union = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
mcp_union

## Crear csvs para análisis

In [43]:
grid_2001_2013 = grid.copy()
grid_2014_2024 = grid.copy()

### Para 2001-2013

In [47]:
promedio_por_celda(f'datos/ndvi_media_2001_2013_reprojected.tif', grid_2001_2013, 'ndvi')
promedio_por_celda(f'datos/lst_reprojected_2001_2013.tif', grid_2001_2013, 'lst')
promedio_por_celda(f'datos/NLCD_reprojected_2011.tif', grid_2001_2013, 'nlcd')
grid_2001_2013['presencia_lobos'] = grid_2001_2013.geometry.intersects(pck_2001.unary_union).astype(int)
df_2001_2013 = grid_2001_2013[['geometry', 'ndvi', 'lst', 'nlcd','presencia_lobos']].copy()
df_2001_2013

/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  med

,geometry,ndvi,lst,nlcd,presencia_lobos
0,"POLYGON ((488217.415 4988588.358, 488217.415 4...",NaN,0.000000,NaN,1
1,"POLYGON ((488217.415 4989088.358, 488217.415 4...",0.097983,0.830405,183.653061,1
2,"POLYGON ((488217.415 4989088.358, 487865.975 4...",0.088706,0.687077,180.173920,1
3,"POLYGON ((488217.415 4988588.358, 488717.415 4...",0.095738,0.309790,190.333328,1
4,"POLYGON ((488217.415 4988588.358, 488217.415 4...",0.082830,2.246931,181.531250,1
...,...,...,...,...,...
36020,"POLYGON ((592217.415 4952588.358, 592217.415 4...",0.048423,-0.864786,181.000000,0
36021,"POLYGON ((592717.415 4950088.358, 592743.919 4...",0.034853,0.000000,NaN,0
36022,"POLYGON ((592717.415 4950088.358, 592717.415 4...",0.028189,-2.088361,172.559998,0
36023,"POLYGON ((592717.415 4950588.358, 592717.415 4...",0.043019,-1.975622,168.352936,0


In [48]:
df_2001_2013.isna().sum()

geometry            0
ndvi               35
lst                 0
nlcd               79
presencia_lobos     0
dtype: int64

In [49]:
df_2001_2013['nlcd'] = df_2001_2013['nlcd'].fillna(df_2001_2013['nlcd'].mean())
df_2001_2013['lst'] = df_2001_2013['lst'].fillna(df_2001_2013['lst'].mean())
df_2001_2013['ndvi'] = df_2001_2013['ndvi'].fillna(df_2001_2013['ndvi'].mean())
df_2001_2013.isna().sum()

geometry           0
ndvi               0
lst                0
nlcd               0
presencia_lobos    0
dtype: int64

In [50]:
df_2001_2013.to_csv('datos/grid_data_2001_2013.csv', index=False)

### Para 2014-2024

In [55]:
promedio_por_celda(f'datos/ndvi_media_2014_2024_reprojected.tif', grid_2014_2024, 'ndvi')
promedio_por_celda(f'datos/lst_reprojected_2014_2024.tif', grid_2014_2024, 'lst')
promedio_por_celda(f'datos/NLCD_reprojected_2021.tif', grid_2014_2024, 'nlcd')
grid_2014_2024['presencia_lobos'] = grid_2014_2024.geometry.intersects(mcp_union.unary_union).astype(int)
df_2014_2024 = grid_2014_2024[['geometry', 'ndvi', 'lst', 'nlcd','presencia_lobos']].copy()
df_2014_2024

/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  media = np.nanmean(data)
/var/folders/lx/01q73krn1md9zqcwvy3rqlqh0000gn/T/ipykernel_42123/2508796881.py:49: RuntimeWarning: Mean of empty slice
  med

,geometry,ndvi,lst,nlcd,presencia_lobos
0,"POLYGON ((488217.415 4988588.358, 488217.415 4...",NaN,0.000000,NaN,0
1,"POLYGON ((488217.415 4989088.358, 488217.415 4...",0.117827,-1.134394,183.653061,0
2,"POLYGON ((488217.415 4989088.358, 487865.975 4...",0.105679,-0.389446,180.173920,0
3,"POLYGON ((488217.415 4988588.358, 488717.415 4...",0.112312,-0.787861,190.333328,1
4,"POLYGON ((488217.415 4988588.358, 488217.415 4...",0.104493,-1.329784,181.531250,1
...,...,...,...,...,...
36020,"POLYGON ((592217.415 4952588.358, 592217.415 4...",0.073449,-2.175720,181.000000,0
36021,"POLYGON ((592717.415 4950088.358, 592743.919 4...",0.062518,0.000000,NaN,0
36022,"POLYGON ((592717.415 4950088.358, 592717.415 4...",0.053039,-3.300823,172.559998,0
36023,"POLYGON ((592717.415 4950588.358, 592717.415 4...",0.065172,-3.122974,168.352936,0


In [56]:
df_2014_2024['nlcd'] = df_2014_2024['nlcd'].fillna(df_2014_2024['nlcd'].mean())
df_2014_2024['lst'] = df_2014_2024['lst'].fillna(df_2014_2024['lst'].mean())
df_2014_2024['ndvi'] = df_2014_2024['ndvi'].fillna(df_2014_2024['ndvi'].mean())
df_2014_2024.isna().sum()

geometry           0
ndvi               0
lst                0
nlcd               0
presencia_lobos    0
dtype: int64

In [57]:
df_2014_2024.to_csv('datos/grid_data_2014_2024.csv', index=False)

In [58]:
df = gpd.GeoDataFrame(df_2014_2024, geometry='geometry')
